# RanPAC Table 1 · Hoàn thiện 2 cấu hình Ablation có Phase 1 (`No RPs` & `NCM with Phase 1`)

Notebook này thuộc bản **tái cài đặt từ mô tả bài báo (Re-implementation)** trên **Split CIFAR-100** (seed `1993`, 10 task × 10 lớp), dùng lại trực tiếp trọng số **AdaptFormer Phase 1** (20 epoch trên task 1) đã huấn luyện ở [04_ranpac_full.ipynb](04_ranpac_full.ipynb) để chạy nốt 2 hàng ablation còn thiếu trong **Table 1**:

| Hàng trong Table 1 | Phase 1 (PETL) | Bộ phân loại Phase 2 | Mốc `A[10]` Paper | Notebook phụ trách |
| --- | --- | --- | ---: | --- |
| `NCM only` | Không | NCM cosine (768-d) | 83.4% | `01_ncm_cosine.ipynb` (**83.38%**) |
| `No RPs or Phase 1` | Không | Ridge không RP (768-d) | 87.0% | `02_ridge_no_rp.ipynb` (**87.02%**) |
| **`NCM with Phase 1`** | **AdaptFormer (Task 1)** | **NCM cosine (768-d)** | **87.8%** | **Notebook 10 (`phase1-ncm`)** |
| `No Phase 1` | Không | Ridge + RP `M=10000` + ReLU | 89.0% | `03_ridge_rp_relu.ipynb` (**88.75%**) |
| **`No RPs`** | **AdaptFormer (Task 1)** | **Ridge không RP (768-d)** | **90.6%** | **Notebook 10 (`phase1-no-rp`)** |
| `Ours (Algorithm 1)` | AdaptFormer (Task 1) | Ridge + RP `M=10000` + ReLU | 92.2% | `04_ranpac_full.ipynb` (**92.02%**) |

Vì cả hai biến thể đều dùng đặc trưng 768 chiều đã thích nghi từ cùng một checkpoint Phase 1 (và tự động khớp khóa `FeatureCache` của Notebook 04 nếu shard vẫn còn trên Drive), mỗi lượt chạy đủ 10 task chỉ mất vài giây đến dưới 1 phút.


## 1. Cài đặt thư viện và kết nối Drive

Đặt `ranpac_core.py`, `ranpac_data.py`, `ranpac_colab.py`, `ranpac_full.py` và `test_ranpac_core.py` trong `MyDrive/RanPAC_implement`. Kết quả log vào project W&B công khai `pnp123/RanPAC-reproduction` với group `cifar100-phase1-table-1` và tag `['reproduction', 'table1']`.


In [ ]:
%pip install -q timm==1.0.22 wandb==0.31.0


In [ ]:
import sys
import unittest
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')
SOURCE_DIR = Path('/content/drive/MyDrive/RanPAC_implement')
WORK_DIR = Path('/content/drive/MyDrive/ranpac-runs')
DATA_ROOT = '/content/ranpac-data'

required = ('ranpac_core.py', 'ranpac_data.py', 'ranpac_colab.py', 'ranpac_full.py', 'test_ranpac_core.py')
missing = [name for name in required if not (SOURCE_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Thiếu {", ".join(missing)} trong {SOURCE_DIR}')
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from ranpac_core import NUM_TASKS, SEED
from ranpac_data import atomic_output
from ranpac_full import PHASE1_ABLATION_PRESETS, run_algorithm1, run_phase1_ablation


In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromName('test_ranpac_core')
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
if not test_result.wasSuccessful():
    raise AssertionError('Kiểm tra unit test trên CPU chưa đạt.')


## 2. Cấu hình checkpoint Phase 1 và thông số chạy

Mặc định trỏ tới `MyDrive/ranpac-runs/runs/full-s1993-main/checkpoint.pt` đã sinh ra từ [04_ranpac_full.ipynb](04_ranpac_full.ipynb). Nếu đặt ở thư mục khác, sửa `ADAPTER_CHECKPOINT_PATH`. Trong trường hợp chạy từ một Drive hoàn toàn mới chưa có checkpoint Phase 1, có thể bật `TRAIN_PHASE1_IF_MISSING = True` để huấn luyện 20 epoch AdaptFormer trước khi chạy hai phép ablation.


In [ ]:
WANDB_ENTITY = 'pnp123'
WANDB_PROJECT = 'RanPAC-reproduction'
WANDB_MODE = 'online'
RUN_LABEL = 'main'

ADAPTER_CHECKPOINT_PATH = WORK_DIR / 'runs' / f'full-s{SEED}-main' / 'checkpoint.pt'
TRAIN_PHASE1_IF_MISSING = False

FEATURE_BATCH_SIZE = 48
NUM_WORKERS = 2
STATS_DEVICE = 'cpu'

if not ADAPTER_CHECKPOINT_PATH.is_file():
    if not TRAIN_PHASE1_IF_MISSING:
        raise FileNotFoundError(
            f'Chưa thấy checkpoint Phase 1 tại {ADAPTER_CHECKPOINT_PATH}. '
            'Hãy kiểm tra đường dẫn của Notebook 04 hoặc đặt TRAIN_PHASE1_IF_MISSING = True.'
        )
    print('Chưa có checkpoint Phase 1 — bắt đầu train 20 epoch AdaptFormer trên Task 1...')
    run_algorithm1(
        WORK_DIR, run_label='phase1-only', resume=False,
        phase1_epochs=20, stop_after_epoch=20, stop_after_task=1,
        feature_batch_size=FEATURE_BATCH_SIZE, stats_device=STATS_DEVICE, workers=NUM_WORKERS,
        data_root=DATA_ROOT, entity=WANDB_ENTITY, project=WANDB_PROJECT, wandb_mode=WANDB_MODE,
        extra_tags=['reproduction', 'table1'],
    )
    ADAPTER_CHECKPOINT_PATH = WORK_DIR / 'runs' / f'full-s{SEED}-phase1-only' / 'checkpoint.pt'

print(f'Sử dụng checkpoint Phase 1: {ADAPTER_CHECKPOINT_PATH}')


## 3. Chạy `NCM with Phase 1` (`phase1-ncm`) và `No RPs` (`phase1-no-rp`)

Hàm `run_phase1_ablation` tự động kiểm tra nếu các shard embedding đã thích nghi có sẵn trong `WORK_DIR/embeddings` thì đọc trực tiếp (0 giây trích đặc trưng); nếu thiếu mới nạp ViT-B/16 + AdaptFormer lên GPU để trích.


In [ ]:
ablation_results = {}
for preset in ('phase1-ncm', 'phase1-no-rp'):
    run_dir = WORK_DIR / 'runs' / f'{preset}-s{SEED}-{RUN_LABEL}'
    ckpt_path = run_dir / 'checkpoint.pt'
    resume_flag = False
    if ckpt_path.is_file():
        saved = torch.load(ckpt_path, map_location='cpu', weights_only=True)
        if len(saved.get('records', ())) == NUM_TASKS:
            print(f'[Đã xong] {run_dir.name} -> A[10] = {saved["records"][-1]["avg_accuracy_pct"]:.2f}%')
            ablation_results[preset] = saved['records']
            continue
        resume_flag = True

    ablation_results[preset] = run_phase1_ablation(
        preset, WORK_DIR,
        adapter_checkpoint_path=ADAPTER_CHECKPOINT_PATH,
        run_label=RUN_LABEL, resume=resume_flag, stop_after_task=NUM_TASKS,
        feature_batch_size=FEATURE_BATCH_SIZE, stats_device=STATS_DEVICE,
        workers=NUM_WORKERS, data_root=DATA_ROOT,
        entity=WANDB_ENTITY, project=WANDB_PROJECT, wandb_mode=WANDB_MODE,
        extra_tags=['reproduction', 'table1'],
    )


## 4. Tổng hợp toàn bộ 6/6 hàng Ablation của Table 1 trên Split CIFAR-100

Cell dưới đọc `A[10]` của các run `01`–`04` từ Drive (nếu có sẵn trong `WORK_DIR/runs`, ngược lại dùng con số đã được xác nhận trong output của các notebook `01`–`04`), kết hợp với 2 kết quả vừa chạy để lập bảng hoàn chỉnh **Table 1 (CIFAR-100)** cùng tỷ lệ giảm lỗi tương đối **Relative Error Rate (`Rel. ER`)** của lớp Random Projection giống cách tính trong bài báo.


In [ ]:
def read_a10_or_fallback(run_folder_name, fallback_pct):
    ckpt_file = WORK_DIR / 'runs' / run_folder_name / 'checkpoint.pt'
    if ckpt_file.is_file():
        data = torch.load(ckpt_file, map_location='cpu', weights_only=True)
        recs = data.get('records') or data.get('tasks') or []
        if len(recs) == NUM_TASKS:
            return float(recs[-1]['avg_accuracy_pct'])
    return fallback_pct


table1_rows = [
    ('NCM only', False, 'NCM cosine', 83.4, read_a10_or_fallback(f'ncm-s{SEED}-main', 83.38)),
    ('No RPs or Phase 1', False, 'Ridge (768-d)', 87.0, read_a10_or_fallback(f'ridge-s{SEED}-main', 87.02)),
    ('No Phase 1', False, 'Ridge + RP (M=10000)', 89.0, read_a10_or_fallback(f'rp-table1-s{SEED}-main', 88.75)),
    ('NCM with Phase 1', True, 'NCM cosine', 87.8, ablation_results['phase1-ncm'][-1]['avg_accuracy_pct']),
    ('No RPs', True, 'Ridge (768-d)', 90.6, ablation_results['phase1-no-rp'][-1]['avg_accuracy_pct']),
    ('Ours (Algorithm 1)', True, 'Ridge + RP (M=10000)', 92.2, read_a10_or_fallback(f'full-s{SEED}-main', 92.02)),
]

df_t1 = pd.DataFrame(
    table1_rows,
    columns=['table1_row', 'phase1_petl', 'classifier', 'paper_a10_pct', 'reimplement_a10_pct'],
)
df_t1['delta_pp'] = df_t1['reimplement_a10_pct'] - df_t1['paper_a10_pct']

def rel_error_reduction(acc_without_rp, acc_with_rp):
    err_before = 100.0 - acc_without_rp
    err_after = 100.0 - acc_with_rp
    return 100.0 * (err_before - err_after) / err_before

er_no_p1 = rel_error_reduction(
    float(df_t1.loc[df_t1['table1_row'] == 'No RPs or Phase 1', 'reimplement_a10_pct'].iloc[0]),
    float(df_t1.loc[df_t1['table1_row'] == 'No Phase 1', 'reimplement_a10_pct'].iloc[0]),
)
er_with_p1 = rel_error_reduction(
    float(df_t1.loc[df_t1['table1_row'] == 'No RPs', 'reimplement_a10_pct'].iloc[0]),
    float(df_t1.loc[df_t1['table1_row'] == 'Ours (Algorithm 1)', 'reimplement_a10_pct'].iloc[0]),
)

summary_dir = WORK_DIR / 'sweeps' / f'table1-cifar100-full-s{SEED}-{RUN_LABEL}'
with atomic_output(summary_dir / 'table1_cifar100_ablation.csv') as tmp:
    df_t1.to_csv(tmp, index=False)

display(df_t1.round(2))
print(f'Rel. ER nhờ RP (không Phase 1): ↓{er_no_p1:.1f}% (Paper báo cáo: ↓15%)')
print(f'Rel. ER nhờ RP (có Phase 1):    ↓{er_with_p1:.1f}% (Paper báo cáo: ↓17%)')

fig, ax = plt.subplots(figsize=(8.5, 4.3), layout='constrained')
heads = ['NCM cosine', 'Ridge (768-d)', 'Ridge + RP (M=10000)']
no_p1_vals = [float(df_t1[(~df_t1['phase1_petl']) & (df_t1['classifier'] == h)]['reimplement_a10_pct'].iloc[0]) for h in heads]
p1_vals = [float(df_t1[(df_t1['phase1_petl']) & (df_t1['classifier'] == h)]['reimplement_a10_pct'].iloc[0]) for h in heads]

x = range(len(heads))
w = 0.34
b1 = ax.bar([i - w / 2 for i in x], no_p1_vals, width=w, label='Không Phase 1 (Frozen ViT)')
b2 = ax.bar([i + w / 2 for i in x], p1_vals, width=w, label='Có Phase 1 (AdaptFormer Task 1)')
ax.bar_label(b1, fmt='%.2f%%', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.2f%%', padding=3, fontsize=9)
ax.set_xticks(list(x), labels=heads)
ax.set_ylim(80, 95)
ax.set_ylabel('A[10] (%) trên Split CIFAR-100')
ax.set_title('Table 1 Factorial Ablation (Re-implement): Phase 1 PETL × Phase 2 Classifier')
ax.legend()
ax.grid(axis='y', alpha=0.25)

with atomic_output(summary_dir / 'table1_cifar100_ablation.png') as tmp:
    fig.savefig(tmp, format='png', dpi=150)
plt.show()
print(f'Đã lưu bảng và biểu đồ Table 1 tại: {summary_dir}')
